## Print information about a tuned chorale
1. Chorale name, tolerance, tonal diamond shape, limit max
1. Cent values, note names, scores, and ratios for every chord
2. Top notes cents, note names, cent values


In [23]:
import os
local_dir = os.getcwd()  # Current working directory
print(f'The local directory is: {local_dir}')

The local directory is: /home/prent/Repos/One-footed-bride-tuning


In [24]:
import logging, os, sys, time
from importlib import reload
import numpy as np
from importlib import reload
from collections import Counter, defaultdict
user = 'prent'
local_dir = os.getcwd()  # Current working directory
print(f'The local directory is: {local_dir}')
base_dir = local_dir
WAVE_DIR = os.path.join(base_dir, 'Music', 'sflib')
# The latest files are here: Archive/straw-man/t1_r1.75_s2.50_md28_sn10/bwv253-opt.npy
numpy_dir = os.path.join(base_dir, 'Archive', 'straw-man')

np.set_printoptions(legacy='1.25')
import diamond_music_utils as dmu
import adaptive_tuning_util as atu 
from itertools import count, combinations, permutations
dmu.start_logger('test.log',log_level = 'info') # how to modify this so that it only prints to the log and not in the notebook.
logging.info(f'{base_dir = }, {numpy_dir = }, {WAVE_DIR = }')
rng = np.random.default_rng()

The local directory is: /home/prent/Repos/One-footed-bride-tuning


In [25]:
print(f'{np.power(2, (1/12)):.6f}')

1.059463


In [26]:
def print_chords(version, input_file, numpy_dir, measure, tolerance, ratios=True, print_individual_chords=True, offset=0, use_werck_top_notes=False, print_top_notes = True):
    
    # load_chorale_in_cents already finds and loads the top-notes file for this version
    # (and re-orders it by chorale frequency), so take top_notes from it rather than
    # rebuilding the path here from a version string that may carry a suffix.
    _, top_notes, chorale, root, mode, keys = atu.load_chorale_in_cents(
        version, numpy_dir, werck_top_notes=use_werck_top_notes)
    try:
        floating_cents = np.load(input_file)
        existing_chorale_in_cents = np.rint(floating_cents).astype(int)
    except:
        print(f'Trouble loading {input_file = }')
        return keys, root, mode
    
    if print_top_notes:
        top_notes = top_notes.copy()
        top_notes[1] = top_notes[1] + offset
        print(f'Key: {keys[root]} {mode}')
        print(f'\ntop notes:')
        print(*[inx for inx in np.arange(12)], sep='\t')
        print(*[note for note in top_notes[0]], sep = '\t')
        print(*[keys[note] for note in top_notes[0]], sep = '\t')
        print(*[cent_value for cent_value in top_notes[1]], sep = '\t')
    if print_individual_chords: 
        print(f'\n#          cents       note names   chord score')
        # #     +----- cents -----+--- note names---+--- chord score'
        # 0:    0  386    0  884\tC♮ E♮ C♮ A♮\t47.0
    if measure > 0: print(f'\nprinting only measure {measure}')
    prev_chord = np.zeros(4, dtype=int)
    header1 = f" # Fr/To Cents Ratio\t # Fr/To Cents Ratio\t # Fr/To Cents Ratio"
    
    for inx, chord_in_cents in zip(count(0,1), existing_chorale_in_cents.T):
        if not np.array_equal(prev_chord, chord_in_cents):
            if measure == 0 or 16 * (measure - 1) <= inx < 16 * measure:
                tuned_pcs = np.array(atu.pitch_class_from_cents(chord_in_cents), dtype=int) % 12
                if print_individual_chords: 
                        # Print tuned note names (from cents), not original MIDI pitch classes.
                        pitches = ' '.join(map(str, keys[tuned_pcs]))
                        print(f'{inx}: {atu.format_chord(chord_in_cents,4)}\t{pitches}\t{chord_scorer.score_chord(chord_in_cents, tolerance=tolerance)}')
                if ratios:
                    print(f'{header1}')
                    intervals = []
                    for inx1, inx2 in combinations(np.arange(4),2):
                            cent_value_interval_pair = np.array([chord_in_cents[inx1], chord_in_cents[inx2]])
                            cent_value_delta, cent_value_moves, cent_value_target = atu.cent_value_interval(cent_value_interval_pair)
                            best_idx = chord_scorer.find_best_interval(cent_value_delta, tolerance)[0]
                            ratio = str(atu.limit_format(tonal_diamond[best_idx])[0]).strip()
                            n1 = keys[tuned_pcs[inx1]]
                            n2 = keys[tuned_pcs[inx2]]
                            intervals.append((n1, n2, cent_value_delta, ratio))

                    def fmt(iv, idx):
                            n1, n2, cents, ratio = iv
                            return f"{idx:>2} {n1:>2} {n2:>2} {cents:>5} {ratio:^6}"

                    # print first and last three intervals on separate lines, nicely aligned and without Python punctuation
                    
                    print("   ".join(fmt(iv, i+1) for i, iv in enumerate(intervals[:3])))
                    print("   ".join(fmt(iv, i+1+3) for i, iv in enumerate(intervals[3:])))
        prev_chord = chord_in_cents.copy()
    return keys, root, mode

In [27]:
suffixes = np.array(['bwv261-opt.npy'])
numpy_dirs = [
'Archive/straw-man/t1_r1.375_lm19',
'Archive/straw-man/t1_r1.25_lm17',
'Archive/straw-man/t2_r1.50_lm19',
'Archive/straw-man/t2_r1.75_lm17',
'Archive/straw-man/t1_r1.75_lm17',
'Archive/straw-man/t3_r1.625_lm19',
'Archive/straw-man/t1_r1.625_lm17',
'Archive/straw-man/t2_r1.25_lm19',
'Archive/straw-man/t2_r1.75_lm19',
'Archive/straw-man/t2_r1.375_lm17',
'Archive/straw-man/t1_r1.625_lm19',
'Archive/straw-man/t3_r1.25_lm19',
'Archive/straw-man/t1_r1.50_lm17',
'Archive/straw-man/t2_r1.50_lm17',
]
# t1_r1.375_lm19
# t1_r1.25_lm17 
# t2_r1.50_lm19 
# t2_r1.75_lm17 
# t1_r1.75_lm17 
# t3_r1.625_lm19
# t1_r1.625_lm17
# t2_r1.25_lm19 
# t2_r1.75_lm19 
# t2_r1.375_lm17
# t1_r1.625_lm19
# t3_r1.25_lm19 
# t1_r1.50_lm17 
# t2_r1.50_lm17 
# t1_r1.75_lm19 
measure = 0 # 0 means print all measures
print_individual_chords = True
ratios = True
print_top_notes = True
print_hits_misses = False
use_werck_top_notes = False
total_scores = 0
num_scores = 0
max_score = 0

for suffix in suffixes:
    for numpy_dir in numpy_dirs:
        # Split the last path component only ('t3_r1.25_lm17'), not the whole path,
        # and convert: '3' is not 3 to build_tonal_diamond or score_chord.
        t, r, lm = os.path.basename(numpy_dir.rstrip('/')).split('_')
        tolerance, ratio_factor, limit_max = int(t[1:]), float(r[1:]), int(lm[2:])
        tonal_diamond = atu.build_tonal_diamond(limit_max)
        chord_scorer = atu.ChordScorer(tonal_diamond)
        chord_scorer.reset_cache()
        print(f'_'*40)
        print(f'{suffix = },  {tolerance = }, {ratio_factor = }, {limit_max = }\n{numpy_dir = },')
        local_numpy_dir = numpy_dir
        # The tuning params now live in the directory name, not the file name:
        # Archive/straw-man/t3_r1.25_lm19/bwv261-opt.npy -> version 'bwv261'
        version = os.path.basename(suffix)[:6]

        try:
            input_file = os.path.join(local_numpy_dir, f'{suffix}') 
            existing_chorale_in_cents = np.load(input_file)
            print(f'Loaded cent file from {input_file}.\nShape is {existing_chorale_in_cents.shape}') # (4,4)
            logging.info(f'{input_file = }')
        except:
            print(f'Trouble loading {input_file = }')
            continue
        num_scores += 1
        scores = np.array([chord_scorer.score_chord(chord, tolerance=tolerance) for chord in existing_chorale_in_cents.T])
        print(f'\nversion: {version}, Tol: {tolerance}, RF: {ratio_factor}, lm: {limit_max}, Average score: {round(np.average(scores),1)}, max score: {np.max(scores)} max chord: {np.argmax(scores)}')
        total_scores += np.average(scores)
        max_score = np.max([max_score, np.max(scores) ])
        keys, root, mode = print_chords(version, input_file, local_numpy_dir, measure, tolerance, \
                ratios=ratios, print_individual_chords=print_individual_chords, \
                use_werck_top_notes=use_werck_top_notes, print_top_notes = print_top_notes)
    if print_hits_misses:
        print(f'hits and misses: {chord_scorer.return_cache_results()}')
    print(f'overall total: {round(total_scores,1)}, {num_scores = }, Average Score: {round(np.average(total_scores/num_scores),1)}, {max_score = }')

________________________________________
suffix = 'bwv261-opt.npy',  tolerance = 1, ratio_factor = 1.375, limit_max = 19
numpy_dir = 'Archive/straw-man/t1_r1.375_lm19',
Loaded cent file from Archive/straw-man/t1_r1.375_lm19/bwv261-opt.npy.
Shape is (4, 280)

version: bwv261, Tol: 1, RF: 1.375, lm: 19, Average score: 57.3, max score: 145.0 max chord: 72
Key: B♮ minor

top notes:
0	1	2	3	4	5	6	7	8	9	10	11
11	2	6	4	1	9	7	10	3	8	5	0
B♮	D♮	F♯	E♮	C♯	A♮	G♮	A♯	D♯	G♯	F♮	C♮
1100	200	600	400	100	900	700	1000	300	800	500	0

#          cents       note names   chord score
0:  199  585  901  199	D♮ F♯ A♮ D♮	43.0
 # Fr/To Cents Ratio	 # Fr/To Cents Ratio	 # Fr/To Cents Ratio
 1 D♮ F♯   386  5/4      2 D♮ A♮   498  4/3      3 D♮ D♮     0  1/1  
 4 F♯ A♮   316  6/5      5 F♯ D♮   386  5/4      6 A♮ D♮   498  4/3  
2:  200  698  902  351	D♮ G♮ A♮ E♮	93.0
 # Fr/To Cents Ratio	 # Fr/To Cents Ratio	 # Fr/To Cents Ratio
 1 D♮ G♮   498  4/3      2 D♮ A♮   498  4/3      3 D♮ E♮   151 12/11 
 4 G♮ A♮   204  9/

In [28]:
print(f'{limit_max = }')
tonal_diamond = atu.build_tonal_diamond(limit_max)
print(f'{tonal_diamond.shape = }')
test_chord = np.array([0, 400, 900, 1100])
print(f'{test_chord = }')
print(f'{chord_scorer.score_chord(test_chord, tolerance=tolerance) = }')
for inx1, inx2 in combinations(np.arange(4),2):
    interval = np.array([test_chord[inx1], test_chord[inx2]])
    interval = atu.cent_value_interval(interval)[0]
    print(f'{interval = }, {dmu.cents_to_ratio(interval, limit_denominator = 50)}')

limit_max = 17
tonal_diamond.shape = (66, 3)
test_chord = array([   0,  400,  900, 1100])
chord_scorer.score_chord(test_chord, tolerance=tolerance) = 3099.0
interval = 400, 63/50
interval = 300, 44/37
interval = 100, 53/50
interval = 500, 4/3
interval = 500, 4/3
interval = 200, 55/49


In [29]:
# Check that cent tunings have not changed the pitch class of any note
print("Checking pitch class preservation...")
violations_found = False
for tolerance in [3]:
    for suffix in suffixes:
            try:
                input_file = os.path.join(local_numpy_dir, f'{suffix}')
                print(f'{input_file = }')
                existing_chorale_in_cents = np.load(input_file)
            except:
                print(f'Could not load {input_file}')
                continue
            version = os.path.basename(suffix)[:6]  # 'bwv261-opt.npy' -> 'bwv261'
            print(f'found {version = }')
            _, _, chorale, root, mode, keys = atu.load_chorale_in_cents(version, local_numpy_dir)

            violations = []
            for chord_inx, (chord_in_cents, chord_12) in enumerate(zip(existing_chorale_in_cents.T, chorale.T)):
                for voice, (cents, midi) in enumerate(zip(chord_in_cents, chord_12)):
                    original_pc = int(midi) % 12
                    tuned_pc = int(atu.pitch_class_from_cents(cents))  # half-up rounding, consistent with horizontal_transpose.py
                    if original_pc != tuned_pc:
                        violations.append((chord_inx, voice, int(midi), cents, original_pc, tuned_pc))

            if violations:
                violations_found = True
                print(f'\n{version}: {len(violations)} pitch class violation(s):')
                for chord_inx, voice, midi, cents, orig_pc, tuned_pc in violations:
                    print(f'  chord {chord_inx}, voice {voice}: MIDI {midi} ({keys[orig_pc]}) -> {cents} cents ({keys[tuned_pc]})')
            else:
                print(f'{version}: OK')

if not violations_found:
    print('\nAll pitch classes preserved across all chorales.')

Checking pitch class preservation...
input_file = 'Archive/straw-man/t2_r1.50_lm17/bwv261-opt.npy'
found version = 'bwv261'
bwv261: OK

All pitch classes preserved across all chorales.


In [30]:
import numpy as np

array_of_durations = np.array([
'05_33',
'09_25',
'07_38',
'05_36',
'11_46',
'07_24',
'03_13',
'04_45',
'03_31',
'04_03',
'06_03',
'02_38',
])

total_seconds = 0
for dur in array_of_durations:
    minutes_str, seconds_str = dur.split('_')  # Split on underscore for reliability
    total_seconds += int(minutes_str) * 60 + int(seconds_str)

# Calculate average in seconds
average_seconds = total_seconds / array_of_durations.shape[0]

# Convert average to minutes and seconds
average_minutes = int(average_seconds // 60)
remaining_seconds = int(average_seconds % 60)

# Format output with zero-padded digits (e.g., "05:03" instead of "5:3")
formatted_average = f"{average_minutes:02d}:{remaining_seconds:02d}"

print(f"Average duration: {formatted_average}")

Average duration: 05:57


In [31]:
# Print the ratios from the root key to each of the notes in each chord.
version = 'bwv264' 
tolerance = 3
limit_max = 17
ratio_factor = '1.500'
# use: Archive/straw-man/viterbi-tunings-5-21/bwv264_t3_r1.500_lm17-trans-sa-opt.npy
numpy_dir = os.path.join('/home', 'prent','Repos', 'One-footed-bride-tuning')
cent_file = os.path.join('Archive', 'straw-man', 'viterbi-tunings-5-21', f'{version}_t{tolerance}_r{ratio_factor}_lm{limit_max}-trans-sa-opt.npy')
print(f'{(cent_file == "Archive/straw-man/viterbi-tunings-5-21/bwv264_t3_r1.500_lm17-trans-sa-opt.npy")}')
input_file = os.path.join(numpy_dir, cent_file)
# Archive/straw-man/viterbi-tunings-5-21/bwv264top-notes.npy
top_note_file = os.path.join(numpy_dir, 'Archive', 'straw-man', 'viterbi-tunings-5-21', f'{version}top-notes.npy')
# returns (chorale_in_cents, top_notes, chorale, root, mode, keys)
_, _, _, root, mode, keys = atu.load_chorale_in_cents(version, numpy_dir)
print(f'Key: {keys[root]} {mode}')
top_notes = np.load(top_note_file)
print(f'{top_notes[0] = }\n{top_notes[1] = }')
root_cent = root * 100
print(f'Root cent value: {root_cent}')
chord_in_cents = np.load(input_file).T
cent_values, cent_counts = np.unique(chord_in_cents, return_counts=True)
print(f'Cent values in chorale: {cent_values.astype(int)}')
print(f'Cent counts in chorale: {cent_counts}') 
print(f'zip of cent values and counts: {list(zip(cent_values.astype(int), cent_counts))}')
    

True
Key: G♮ major
top_notes[0] = array([ 7,  2, 11,  9,  6,  0,  4,  3,  1,  5,  8, 10])
top_notes[1] = array([ 700,  200, 1100,  900,  600,    0,  400,  300,  100,  500,  800,
       1000])
Root cent value: 700
Cent values in chorale: [   0    2    3    8   97  105  189  190  198  200  201  203  204  206
  207  208  223  224  276  364  365  372  373  379  386  388  389  394
  576  584  590  592  593  595  610  618  680  688  689  695  696  698
  699  701  702  704  705  706  710  714  722  884  892  900  906  908
  909  926  947 1066 1074 1075 1082 1084 1085 1087 1088 1090 1091 1092
 1108 1158 1172 1175 1192 1193]
Cent counts in chorale: [ 8  2  8  8  1  1  1 14 32  4 12 26 16 21 14  8  1 12  4  3  2  2  2  4
  7  6  4  4  2 12  7 10  2  1  4  2  8  5  4  2 16  8 24 36 16 12 10 16
  2  1  8  4  8 12  8  8  4  8  2  2  4  2  8  4 12 20  8 18  4  8  4  1
  2  1  1  4]
zip of cent values and counts: [(0, 8), (2, 2), (3, 8), (8, 8), (97, 1), (105, 1), (189, 1), (190, 14), (198, 32), (200